# ランク80：座標を事前準備した従来法と提案法

同じ「123」の1シート（2048²、43モード×41 Thread、ランク80）。同じ固定41座標の事前準備を両方式に認めます。

- **legacy**：各Threadを同軸コア加算→座標入りの空間因子で復元→フルサイズ画像加算。FFTもオンラインのシフトも呼びません。
- **proposed**：シフト行列も事前準備→コア移動・加算→最後に1回復元。
- **proposed_online**：従来の提案法の計測条件。シフト行列をオンラインで計算する参考値。

元の14.65秒は復元画像の移動を含んでいたため、新しい比較には使いません。上から再実行してください。

コア・基の空間因子は共通の保存済み辞書から読みます。従来の空間Tucker学習を再現する比較ではありません。従来法の座標反映は因子の行への線形補間、提案法はFourier射影。画質指標も併記します。

**準備時間・座標用キャッシュ容量は別記録**。初期値は3回測定・ウォームアップ0回。CGH化・伝搬は含みません。PSNR/SSIMはCGH化前の全画素強度です。


In [1]:
import importlib
from datetime import datetime
from pathlib import Path
import lsh_digit1_proposed as proposed
import legacy_vs_proposed_rank80 as comparison
proposed = importlib.reload(proposed)
comparison = importlib.reload(comparison)

RANK = 80
REPEATS = 3  # とにかく1回だけ測る場合は1。
WARMUPS = 0
OUTPUT_DIR = Path(f'legacy_vs_proposed_rank{RANK}_outputs_fixed_{datetime.now():%Y%m%d_%H%M%S_%f}')
summary = comparison.run_benchmark(
    output_dir=OUTPUT_DIR, rank=RANK, repeats=REPEATS, warmups=WARMUPS,
)


legacy 1/3: 1.4228 s
proposed 1/3: 0.0377 s
proposed_online 1/3: 0.1623 s
proposed 2/3: 0.0345 s
proposed_online 2/3: 0.1522 s
legacy 2/3: 1.3849 s
proposed_online 3/3: 0.1527 s
legacy 3/3: 1.3141 s
proposed 3/3: 0.0311 s

方式 | 場生成[s] | コア重み付け | 座標準備(オンライン) | 加算(コア移動を含む) | 復元
legacy | 1.3849 | 0.0142 | 0.0000 | 0.3227 | 1.0474
proposed | 0.0345 | 0.0072 | 0.0000 | 0.0061 | 0.0210
proposed_online | 0.1527 | 0.0091 | 0.1160 | 0.0093 | 0.0200
従来法/提案法の時間比: 40.09
直接場への複素場誤差[%]: {'proposed_online': 96.81454502380433, 'legacy': 96.80812443541559, 'proposed': 96.81454502380433}

CGH化前の強度画像（全画素）: 比較 | SSIM | PSNR[dB]
legacy_vs_direct | 0.916675 | 31.582
proposed_vs_direct | 0.916656 | 31.582
proposed_vs_legacy | 0.999831 | 71.476
proposed_online_vs_direct | 0.916656 | 31.582
保存先: C:\Users\kota-\Github\Project_new\research\legacy_vs_proposed_rank80_outputs_fixed_20261006_123920_039204


In [2]:
print('時間の比較表:', (OUTPUT_DIR / 'timing_comparison.csv').resolve())
print('条件・時間内訳・出力誤差:', (OUTPUT_DIR / 'summary.json').resolve())
print('事前の読込・辞書詰め替え[秒]:', summary['dictionary_load_and_packing_seconds'])
print('両方式で共通の辞書容量[MiB]:', summary['dictionary_array_mib'])
print('提案法対従来法の複素場誤差:', summary['complex_relative_l2_proposed_vs_legacy'])
print('座標の事前準備時間[秒]:', summary['coordinate_preparation_seconds'])
print('座標用キャッシュ[MiB]:', summary['coordinate_cache_mib'])
print('SSIM/PSNR CSV:', (OUTPUT_DIR / 'quality_comparison.csv').resolve())


時間の比較表: C:\Users\kota-\Github\Project_new\research\legacy_vs_proposed_rank80_outputs_fixed_20261006_123920_039204\timing_comparison.csv
条件・時間内訳・出力誤差: C:\Users\kota-\Github\Project_new\research\legacy_vs_proposed_rank80_outputs_fixed_20261006_123920_039204\summary.json
事前の読込・辞書詰め替え[秒]: 1.049424100012402
両方式で共通の辞書容量[MiB]: 11.69921875
提案法対従来法の複素場誤差: 0.04768172121170371
座標の事前準備時間[秒]: {'legacy_placed_factors': 0.2552692000026582, 'proposed_shift_operators': 0.13799869999638759}
座標用キャッシュ[MiB]: {'legacy_placed_factors': 102.5, 'proposed_shift_operators': 4.00390625}
SSIM/PSNR CSV: C:\Users\kota-\Github\Project_new\research\legacy_vs_proposed_rank80_outputs_fixed_20261006_123920_039204\quality_comparison.csv


In [3]:
# 画質のみ追加：このセルだけで実行できる。
import importlib
from pathlib import Path
import lsh_digit1_proposed as proposed
import legacy_vs_proposed_rank80 as comparison
proposed = importlib.reload(proposed)
comparison = importlib.reload(comparison)
runs = sorted(Path('.').glob('legacy_vs_proposed_rank80*/summary.json'), key=lambda p: p.stat().st_mtime_ns)
if not runs:
    raise FileNotFoundError('No benchmark result found')
SAVED_RUN = runs[-1].parent
quality = comparison.evaluate_saved_run(SAVED_RUN)
print('SSIM/PSNR CSV:', (SAVED_RUN / 'quality_comparison.csv').resolve())



CGH化前の強度画像（全画素）: 比較 | SSIM | PSNR[dB]
legacy_vs_direct | 0.916675 | 31.582
proposed_vs_direct | 0.916656 | 31.582
proposed_vs_legacy | 0.999831 | 71.476
proposed_online_vs_direct | 0.916656 | 31.582
SSIM/PSNR CSV: C:\Users\kota-\Github\Project_new\research\legacy_vs_proposed_rank80_outputs_fixed_20261006_123920_039204\quality_comparison.csv
